In [1]:
import pandas as pd

In [2]:
import numpy as np
import requests
from datetime import datetime

In [3]:
data = pd.read_csv("TNweather_1.8M[1].csv")

In [4]:
data

,Unnamed: 0,temperature_2m,relative_humidity_2m,dew_point_2m,precipitation,rain,surface_pressure,cloud_cover,cloud_cover_low,wind_speed_10m,wind_direction_10m,time,city,rain_tomorrow
0,0,23.0,94.0,22.0,0.0,0.0,1004.8,88.0,29.0,8.2,23.0,2020-01-01 00:00:00,Ariyalur,1
1,1,22.8,95.0,22.0,0.0,0.0,1004.2,65.0,29.0,8.0,8.0,2020-01-01 01:00:00,Ariyalur,1
2,2,22.8,96.0,22.1,0.0,0.0,1003.5,66.0,25.0,8.3,360.0,2020-01-01 02:00:00,Ariyalur,1
3,3,22.6,97.0,22.1,0.0,0.0,1003.0,92.0,76.0,8.6,358.0,2020-01-01 03:00:00,Ariyalur,1
4,4,22.6,97.0,22.1,0.0,0.0,1003.3,100.0,100.0,9.0,360.0,2020-01-01 04:00:00,Ariyalur,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1888417,1888417,28.5,53.0,18.1,0.0,0.0,997.4,100.0,0.0,6.3,340.0,2025-08-25 07:00:00,Virudhunagar,1
1888418,1888418,31.1,42.0,16.8,0.0,0.0,997.6,99.0,0.0,5.9,337.0,2025-08-25 08:00:00,Virudhunagar,1
1888419,1888419,33.6,36.0,16.5,0.0,0.0,997.3,80.0,0.0,3.9,336.0,2025-08-25 09:00:00,Virudhunagar,1
1888420,1888420,35.5,32.0,16.5,0.0,0.0,997.0,61.0,0.0,1.6,27.0,2025-08-25 10:00:00,Virudhunagar,1


In [5]:
data["time"] = pd.to_datetime(data["time"])

In [6]:
data = data[
    ["time",
     "temperature_2m",
     "relative_humidity_2m",
     "precipitation",
     "cloud_cover"]
]

In [7]:
data["harvest_label"] = (
    (data["precipitation"] == 0) &
    (data["relative_humidity_2m"] < 80) &
    (data["temperature_2m"].between(20, 35))
).astype(int)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_26352\3583109768.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["harvest_label"] = (


In [8]:
X_train = data[
    ["temperature_2m",
     "relative_humidity_2m",
     "precipitation",
     "cloud_cover"]
]

y_train = data["harvest_label"]

In [9]:
from sklearn.ensemble import RandomForestClassifier

In [10]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [35]:
future_dates = pd.date_range(
    start="2026-01-01",
    end="2026-12-31",
    freq="D"
)

future = pd.DataFrame({"time": future_dates})

In [36]:
future["month"] = future["time"].dt.month
future["day"] = future["time"].dt.day

data["month"] = data["time"].dt.month
data["day"] = data["time"].dt.day


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [37]:
avg_weather = data.groupby(["month","day"])[
    ["temperature_2m",
     "relative_humidity_2m",
     "precipitation",
     "cloud_cover"]
].mean().reset_index()

In [38]:
future = pd.merge(
    future,
    avg_weather,
    on=["month","day"],
    how="left"
)

In [39]:
today = pd.to_datetime(datetime.today().date())

future = future[future["time"] >= today]

In [40]:
X_future = future[X_train.columns]

In [41]:
X_future


,temperature_2m,relative_humidity_2m,precipitation,cloud_cover
57,26.745833,63.925073,0.012646,49.807200
58,26.518878,65.110197,0.057219,44.887975
59,26.798648,64.612756,0.057931,57.535636
60,26.736038,61.453947,0.018732,59.483187
61,26.835069,60.686221,0.006067,49.039656
...,...,...,...,...
360,24.330351,77.851096,0.125570,49.020395
361,24.674408,76.548026,0.033904,55.100439
362,24.671228,77.665570,0.063114,69.105482
363,24.527610,79.360526,0.130461,69.091447


In [42]:
future["harvest_prediction"] = model.predict(X_future)


In [43]:
today = pd.to_datetime(datetime.today().date())

In [44]:
future = future[future["time"] >= today]

In [45]:
harvest_days = future[future["harvest_prediction"] == 1].copy()

harvest_days["gap"] = harvest_days["time"].diff().dt.days

harvest_days["window"] = (harvest_days["gap"] > 1).cumsum()

windows = harvest_days.groupby("window")["time"].agg(["min", "max"]).reset_index()

In [46]:
windows["days"] = (windows["max"] - windows["min"]).dt.days + 1
windows = windows[windows["days"] >= 3]

In [47]:
if len(windows) > 0:
    next_window = windows.iloc[0]

    print("🌾 Recommended Harvest Window")
    print(f"📅 {next_window['min'].date()} → {next_window['max'].date()}")
else:
    print("No upcoming harvest window found")

🌾 Recommended Harvest Window
📅 2026-03-02 → 2026-03-06


In [48]:
window_data = future[
    (future["time"] >= next_window["min"]) &
    (future["time"] <= next_window["max"])
]

In [49]:
X_window = window_data[X_train.columns]

In [50]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_window)

In [51]:
importance = np.abs(shap_values[1]).mean(axis=0)

feature_importance = dict(zip(X_window.columns, importance))

sorted_features = sorted(
    feature_importance.items(),
    key=lambda x: x[1],
    reverse=True
)

In [52]:
if len(windows) > 0:
    next_window = windows.iloc[0]

    print("🌾 Recommended Harvest Window")
    print(f"📅 {next_window['min'].date()} → {next_window['max'].date()}")
else:
    print("⚠ No suitable harvest window found in upcoming days.")

print("\n💡 Why this window is suitable:")

top_features = [f[0] for f in sorted_features[:3]]

for f in top_features:
    if f == "precipitation":
        print("• No rainfall expected")
    elif f == "relative_humidity_2m":
        print("• Moderate humidity prevents crop spoilage")
    elif f == "temperature_2m":
        print("• Ideal temperature protects crop quality")
    elif f == "cloud_cover":
        print("• Clear weather reduces rain risk")

🌾 Recommended Harvest Window
📅 2026-03-02 → 2026-03-06

💡 Why this window is suitable:
• Moderate humidity prevents crop spoilage
• No rainfall expected
• Ideal temperature protects crop quality
